# Notebook 07: Gemma 3 Integration and Prompt Engineering

## Objective

Integrate a pretrained Gemma 3 instruction-tuned model
into EdoFarm AI.

This notebook will:

1. Establish hosted model inference.
2. Configure the model interface.
3. Develop the agricultural system prompt.
4. Define language and response instructions.
5. Establish agricultural safety constraints.
6. Run initial English and Edo response tests.
7. Record model responses for evaluation.

## Selected model

google/gemma-3-4b-it

## Inference provider

Hugging Face Inference Providers.

## Important limitations

- No fine-tuning is performed in this notebook.
- The model is not assumed to have verified Edo language ability.
- No unverified Q&A candidates are treated as approved knowledge.
- No pesticide rates or unsupported treatment schedules should
  be invented.
- The intent classifier is not yet connected.
- Retrieval integration will be implemented separately.

In [1]:
%pip install -U huggingface_hub python-dotenv pandas

  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os
import json
import pandas as pd
from datetime import datetime, timezone

def find_project_root(start_path=Path.cwd()):
    for path in [start_path, *start_path.parents]:
        if (
            (path / "data" / "processed").is_dir()
            and (path / "notebooks").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate EdoFarm AI project root."
    )

ROOT = find_project_root()

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "interim"

GEMMA_OUTPUT_DIR = (
    INTERIM_DIR / "gemma_evaluation"
)

GEMMA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_ID = "google/gemma-3-4b-it"

print("Project root:", ROOT)
print("Gemma model:", MODEL_ID)
print("Evaluation directory:", GEMMA_OUTPUT_DIR)

Project root: c:\Users\XXMUGENX\3D Objects\edofarm-ai
Gemma model: google/gemma-3-4b-it
Evaluation directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\interim\gemma_evaluation


## 1. Configure Hugging Face Authentication

Load the Hugging Face access token from the project's
environment configuration.

The token is never hardcoded into notebook cells
or committed to version control.

In [3]:
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv(ROOT / ".env", override=False)

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found. Add it to the project root .env file."
    )

gemma_client = InferenceClient(
    token=HF_TOKEN
)

print("Hugging Face token detected.")
print("Inference client initialized.")
print("Model:", MODEL_ID)

Hugging Face token detected.
Inference client initialized.
Model: google/gemma-3-4b-it


## 2. EdoFarm AI System Prompt

Define the model's role, intended agricultural scope,
language requirements and safety boundaries.

The prompt must distinguish between approved source information
and general educational explanations.

When no approved knowledge context is supplied, the model
must not present specific agricultural recommendations
as source-verified advice.

In [4]:
SYSTEM_PROMPT = """
You are EdoFarm AI, an agricultural information assistant
designed for farmers in Edo State and other Nigerian communities.

YOUR ROLE

Help users understand crop production, agricultural practices
and common crop-related problems using clear, accessible language.

SUPPORTED AGRICULTURAL SCOPE

You may assist with:

1. Crop cultivation.
2. Pest management.
3. Disease management.
4. Soil management.
5. Irrigation.
6. Fertilizer application.
7. Harvesting.

Livestock, poultry, veterinary medicine and aquaculture
are outside your current scope.

EVIDENCE AND KNOWLEDGE RULES

1. Treat supplied APPROVED KNOWLEDGE CONTEXT as the trusted
   project knowledge for specific agricultural claims.

2. Never invent sources, research findings, statistics or
   document references.

3. Distinguish general educational explanations from
   source-supported agricultural recommendations.

4. When approved context is unavailable or insufficient,
   clearly explain this limitation.

5. Do not pretend that an unsupported recommendation has
   been verified for Edo State.

AGRICULTURAL SAFETY

1. Do not invent pesticide or herbicide product names,
   application rates, dosages or treatment schedules.

2. Do not give fertilizer application rates unless supported
   by approved, relevant source material.

3. Do not diagnose a crop disease with certainty from a brief
   description.

4. Ask follow-up questions when crop type, symptoms, location
   or farming conditions are insufficiently described.

5. For serious outbreaks or uncertain chemical treatments,
   recommend contacting a qualified agricultural extension
   officer.

6. Never claim that a treatment is guaranteed to work.

LANGUAGE

Respond in the language requested by the user.

Supported prototype response modes:

- English.
- Edo (Bini).

For Edo (Bini), use your best available language capability.
Do not invent words when uncertain.

Edo responses are provisional and have not been validated
by native Edo speakers.

If you cannot produce a reliable Edo response, explain
that language review is needed.

RESPONSE STYLE

1. Be conversational and respectful.
2. Use simple language.
3. Avoid unnecessary technical jargon.
4. Organize longer explanations into short sections.
5. Give practical next steps only when adequately supported.
6. Ask concise clarification questions when necessary.

IMPORTANT

You are an informational assistant, not a replacement
for agricultural experts or extension officers.
"""

In [5]:
def generate_gemma_response(
    question,
    language="English",
    approved_context=None,
    max_tokens=350
):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")

    if language not in ["English", "Edo (Bini)"]:
        raise ValueError("Unsupported response language.")

    context_text = (
        approved_context.strip()
        if approved_context
        else "NO APPROVED KNOWLEDGE CONTEXT PROVIDED."
    )

    user_prompt = f"""
REQUESTED RESPONSE LANGUAGE: {language}

APPROVED KNOWLEDGE CONTEXT:
{context_text}

FARMER QUESTION:
{question}

INSTRUCTIONS:

Answer the farmer's question in the requested language.

Use the supplied approved context when available.

If no approved context is supplied:
- You may offer a cautious general educational explanation.
- Explicitly distinguish it from locally verified advice.
- Do not invent specific agricultural application rates,
  chemical recommendations or treatment schedules.

Do not fabricate sources or citations.
"""

    try:
        response = gemma_client.chat_completion(
            model=MODEL_ID,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            max_tokens=max_tokens,
            temperature=0.3
        )

        generated_text = (
            response.choices[0].message.content
        )

        if not generated_text:
            raise RuntimeError("Gemma returned an empty response.")

        return {
            "model": MODEL_ID,
            "requested_language": language,
            "question": question,
            "response": generated_text.strip(),
            "approved_context_used": bool(approved_context),
            "status": "success"
        }

    except Exception as error:
        return {
            "model": MODEL_ID,
            "requested_language": language,
            "question": question,
            "response": None,
            "approved_context_used": bool(approved_context),
            "status": "failed",
            "error": str(error)
        }


print("Gemma response function defined.")

Gemma response function defined.


In [7]:
english_test = generate_gemma_response(
    question=(
        "Why is it important to test soil before planting "
        "a crop? Explain in simple terms."
    ),
    language="English"
)

print("STATUS:", english_test["status"])
print("\nQUESTION:")
print(english_test["question"])

print("\nGEMMA RESPONSE:")
print(english_test["response"])

if english_test["status"] != "success":
    print("\nERROR:")
    print(english_test.get("error"))

STATUS: failed

QUESTION:
Why is it important to test soil before planting a crop? Explain in simple terms.

GEMMA RESPONSE:
None

ERROR:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6b01e964-77e25e6c3dc0ddc2605c5d71;285d72f9-2b43-426a-9a86-86548bbc1238)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.


## Why We Stopped Using Gemma

Gemma was initially considered as the language model for generating agricultural advisory responses in EdoFarm AI.

During implementation, we encountered an HTTP `402 Payment Required` error when attempting to access Gemma through the Hugging Face Inference Providers API.

### What Happened?

The request was sent to:

`https://router.huggingface.co/v1/chat/completions`

However, Hugging Face returned the following error:

```text
402 Payment Required

Purchase pre-paid credits to continue using Inference Providers.
Alternatively, subscribe to PRO to get monthly included credits.